<a href="https://colab.research.google.com/github/ScreaM835/QNM_Project_Fresh/blob/main/notebooks/hybrid_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid coarse-FD + FNO residual surrogate — Colab (A100) run

End-to-end reproduction of the Schwarzschild hybrid surrogate on a Colab GPU, using the **exact deployed configuration** (`configs/hybrid_sw_gate_s1em3.yaml`, batch 8, 100 epochs). This notebook only *calls* the repository scripts — it does **not** modify any core code.

**Before running:** `Runtime ▸ Change runtime type ▸ A100 GPU` (Pro+). An L4 (24 GB) or V100 (16 GB) also work — only the 12 GB laptop GPU did not, because batch-8 activations at 501×1001 resolution do not fit in 12 GB.

Pipeline: **clone → install → build corpus → train → evaluate (incl. observer-position sweep) → download results.**

Reproduction targets from the paper: validation MSE floor ≈ `4.0e-7`, canonical RMSD ≈ `6.85e-4`, population field-MSE ratio ≈ `79×`.

## 1 · Confirm the GPU
Want an A100 (40 GB). If you see a smaller card, that is fine as long as it has ≥ 16 GB.

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07


## 2 · Clone the public repository

In [2]:
import os

REPO_URL = "https://github.com/ScreaM835/QNM_Project_Fresh.git"
REPO_DIR = "/content/QNM_Project_Fresh"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --oneline

Cloning into '/content/QNM_Project_Fresh'...
remote: Enumerating objects: 888, done.
remote: Counting objects: 100% (888/888), done.
remote: Compressing objects: 100% (771/771), done.
remote: Total 888 (delta 107), reused 860 (delta 104), pack-reused 0 (from 0)
Receiving objects: 100% (888/888), 121.91 MiB | 14.45 MiB/s, done.
Resolving deltas: 100% (107/107), done.
/content/QNM_Project_Fresh
fac0d06 (grafted, HEAD -> main, origin/main, origin/HEAD) Add Colab (A100) notebook for hybrid surrogate reproduction + observer-position sweep (calls existing scripts only, no core-code changes)


## 3 · Install dependencies
`neuraloperator` is pinned to **2.0.0**: the hybrid model only builds correctly on this version (a newer release changes the FNO API). Colab already ships a compatible PyTorch, so we do not reinstall torch.

In [3]:
!pip -q install "neuraloperator==2.0.0" "numpy>=1.24" "scipy>=1.10" "pyyaml>=6.0" "tqdm>=4.66"

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.6/248.6 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 7.6 MB/s eta 0:00:00
torch 2.11.0+cu128 | CUDA available: True
device: NVIDIA A100-SXM4-40GB


## 4 · Build the coarse/fine FD corpus
Regenerated on Colab (deterministic, seed 0) rather than uploaded — the build is CPU-only and takes a few minutes. Produces the `k=4` prior corpus and the `k=2` corpus used to construct the label-free Richardson target.

In [4]:
!python scripts/build_hybrid_dataset.py --config configs/hybrid_sw_dataset.yaml --k 4 --out outputs/hybrid/dataset_sw_k4.npz
!python scripts/build_hybrid_dataset.py --config configs/hybrid_sw_dataset.yaml --k 2 --out outputs/hybrid/dataset_sw_k2.npz

[HYBRID-DATA] k=4, total samples=1200 (train=1000, val=100, test=100)
/content/QNM_Project_Fresh/src/hybrid_dataset.py:60: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  pts = sampler.random(n=n)
  train 1/1000: M=1.140 x0=+5.73 sigma=4.45
  train 2/1000: M=0.822 x0=+3.62 sigma=6.56
  train 3/1000: M=0.934 x0=+4.44 sigma=3.02
  train 4/1000: M=1.003 x0=+2.34 sigma=5.97
  train 5/1000: M=1.089 x0=+4.98 sigma=6.17
  train 6/1000: M=0.971 x0=+2.86 sigma=4.84
  train 7/1000: M=0.883 x0=+5.20 sigma=5.36
  train 8/1000: M=1.152 x0=+3.08 sigma=3.63
  train 9/1000: M=1.180 x0=+4.12 sigma=5.06
  train 10/1000: M=0.861 x0=+2.22 sigma=3.95
  train 11/1000: M=0.999 x0=+5.84 sigma=6.48
  train 12/1000: M=1.067 x0=+3.94 sigma=4.52
  train 13/1000: M=1.031 x0=+5.36 sigma=3.34
  train 14/1000: M=0.912 x0=+3.48 sigma=5.66
  train 15/1000: M=0.850 x0=+4.58 sigma=4.13
  train 16/1000: M=1.118 x0=+2.70 sigma=6.86
  train 17/1000: M=1.108 x0=+4.73 sigma=3.85
  train 18

## 5 · Train the hybrid surrogate
Exact deployed config: gate scale `s = 1e-3`, label-free Richardson target, batch 8, 100 Adam epochs, seed 1234. On an A100 this is well under an hour. Writes `model_best.pt`, `model_last.pt`, and the training history to `outputs/hybrid/fno_sw_gate_s1em3/`.

In [5]:
!python scripts/train_hybrid_fno.py --config configs/hybrid_sw_gate_s1em3.yaml

[HYBRID-TRAIN] loading k4=outputs/hybrid/dataset_sw_k4.npz  k2=outputs/hybrid/dataset_sw_k2.npz  (target_mode=richardson)
[HYBRID-TRAIN] meta k4: {'k': 4, 'base_dx': 0.2, 'base_dt': 0.1, 'coarse_dx': 0.8, 'coarse_dt': 0.4, 'sweep': {'M_range': [0.8, 1.2], 'x0_range': [2.0, 6.0], 'sigma_range': [3.0, 7.0], 'n_train': 1000, 'n_val': 100, 'n_test': 100, 'seed': 0}, 'smoke': False}
[HYBRID-TRAIN] meta k2: {'k': 2, 'base_dx': 0.2, 'base_dt': 0.1, 'coarse_dx': 0.4, 'coarse_dt': 0.2, 'sweep': {'M_range': [0.8, 1.2], 'x0_range': [2.0, 6.0], 'sigma_range': [3.0, 7.0], 'n_train': 1000, 'n_val': 100, 'n_test': 100, 'seed': 0}, 'smoke': False}
[HYBRID-TRAIN] assembled in 104.8s: X_tr (1000, 5, 501, 1001), Y_tr (1000, 1, 501, 1001), X_va (100, 5, 501, 1001)
[HYBRID-TRAIN] loss weights (data,physics,anchor)=(1.0, 0.0, 0.0)  dx_f=0.200 dt_f=0.100
[HYBRID-TRAIN] model params: 1127521
Traceback (most recent call last):
  File "/content/QNM_Project_Fresh/scripts/train_hybrid_fno.py", line 497, in <modul

## 6 · Evaluate — field accuracy + QNM extraction
Runs the 100-BH test-set field metrics and the M1–M5 extraction. The `--xq` flag drives the **observer-position sweep**: here we extract at `x_q ∈ {2, 5, 10, 15, 20} M` (against the FD reference at the same columns) on the fixed `[10, 50] M` window, to test whether a more asymptotic observer is preferable to `x_q = 2 M` given that the time domain was not extended.

In [6]:
!python scripts/eval_hybrid_sw.py --config configs/hybrid_sw_gate_s1em3.yaml --xq 2 5 10 15 20 --t_start 10 --t_end 50

No checkpoint at outputs/hybrid/fno_sw_gate_s1em3/model_best.pt


## 7 · Package the results for download
Zips the trained model, history, evaluation reports and figures. Download the archive, unzip into `outputs/hybrid/fno_sw_gate_s1em3/` in your local workspace, and the report figures/tables regenerate as normal.

In [7]:
import shutil, os

src = "outputs/hybrid/fno_sw_gate_s1em3"
archive = "/content/hybrid_sw_gate_s1em3_colab"
shutil.make_archive(archive, "zip", src)
print("created", archive + ".zip",
      f"({os.path.getsize(archive + '.zip')/1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(archive + ".zip")
except Exception as e:
    print("Auto-download unavailable; find the zip in the file browser at", archive + ".zip", "|", e)

created /content/hybrid_sw_gate_s1em3_colab.zip (2.4 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Optional · Save straight to Google Drive
If you prefer Drive over a browser download, run this instead of (or after) cell 7.

In [8]:
from google.colab import drive
drive.mount("/content/drive")
import shutil
shutil.copy("/content/hybrid_sw_gate_s1em3_colab.zip",
            "/content/drive/MyDrive/hybrid_sw_gate_s1em3_colab.zip")
print("copied to Drive: MyDrive/hybrid_sw_gate_s1em3_colab.zip")

Mounted at /content/drive
copied to Drive: MyDrive/hybrid_sw_gate_s1em3_colab.zip
